# VQE and VQD optimizer benchmark for a single transmon

This notebook extends the supplied **VQE/VQD Single Transmon Simulation** into a reproducible optimizer benchmark aimed at eventual quantum-hardware execution. It preserves the original single-mode charge Hamiltonian and physical parameters: $L_J=10$ nH, $f_{\rm lin}=5.21015$ GHz, $n_g=0$, and charge cutoff $N=8$ (17 physical charge states embedded in 5 qubits).

The benchmark compares COBYLA, Nelder–Mead, SPSA, and Adam/SPSA-gradient updates under:

- exact statevector expectation values;
- finite-shot Pauli measurements with a **total** shot budget per objective call;
- an optional lightweight readout/depolarizing-noise emulator;
- repeated random seeds and the same objective-evaluation budget;
- VQE ground-energy error and VQD first-excited-state/transition error;
- variance, convergence, objective calls, total shots, and approximate circuit cost.

The exact diagonalization is used only as a validation reference. The noisy estimator samples each Pauli expectation and does not inject Gaussian noise into the final energy. This is a portable, dependency-light model—not a pulse-level or device-calibrated simulator. See the final hardware recommendations.


In [1]:
# Core dependencies. Plotly is used because it works in both Jupyter and Colab.
import math, time, warnings
from dataclasses import dataclass
from collections import defaultdict
import numpy as np
from scipy.constants import e as electron_charge, h as planck_h
from scipy.optimize import minimize
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

np.set_printoptions(precision=5, suppress=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)
print("NumPy", np.__version__)

def show_figure(fig):
    # Avoid Plotly's MIME renderer, which requires nbformat in some Qiskit envs.
    display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))


NumPy 2.4.6


## 1. Preserve the original physics

The charge-basis Hamiltonian is

$$H_{nm}=4E_C(n-n_g)^2\delta_{nm}-\frac{E_J}{2}(\delta_{n,m+1}+\delta_{n,m-1}),$$

with energies expressed as frequency in GHz. As in the supplied notebook, the 17-state physical block is embedded in a 32-dimensional five-qubit space; padding states are assigned a safe high energy.


In [2]:
L_J_nH, f_linear_GHz, n_g, N = 10.0, 5.21015, 0.0, 8
L_J = L_J_nH * 1e-9
C_effective = 1 / (L_J * (2*np.pi*f_linear_GHz*1e9)**2)
E_C = electron_charge**2 / (2*C_effective*planck_h) / 1e9
E_J = (planck_h/(4*np.pi*electron_charge))**2 / (L_J*planck_h) / 1e9
n = np.arange(-N, N+1); d = len(n); q = int(d-1).bit_length(); D = 2**q
H = np.diag(4*E_C*(n-n_g)**2) + np.diag(np.full(d-1, -E_J/2), 1) + np.diag(np.full(d-1, -E_J/2), -1)
levels, vectors = np.linalg.eigh(H)
E0_exact, E1_exact = map(float, levels[:2]); f01_exact = E1_exact-E0_exact
E_pad = float(4*E_C*N*N + E_J)
A = E_pad*np.eye(D); A[:d,:d] = H
print(f"Ceff={C_effective*1e15:.6f} fF, EC/h={E_C:.9f} GHz, EJ/h={E_J:.9f} GHz")
print(f"{d} charge states on {q} qubits; exact E0={E0_exact:.9f}, E1={E1_exact:.9f}, f01={f01_exact:.9f} GHz")


Ceff=93.312497 fF, EC/h=0.207584515 GHz, EJ/h=16.346151281 GHz
17 charge states on 5 qubits; exact E0=-13.794074163, E1=-8.800796285, f01=4.993277878 GHz


## 2. Pauli measurement model and hardware-efficient ansatz

The original VQE ansatz is retained conceptually: four layers of $R_Y$ rotations with circular CNOT entanglers. A small statevector implementation avoids dependence on a particular Qiskit release. The Pauli decomposition is exact. Under finite shots, each non-identity Pauli term receives shots approximately proportional to $|c_P|\sqrt{1-\langle P\rangle^2}$, with a pilot-free coefficient fallback. The configured number is the **total shots across all Pauli terms**, not shots per term.


In [3]:
I=np.eye(2); X=np.array([[0,1],[1,0]],complex); Y=np.array([[0,-1j],[1j,0]],complex); Z=np.diag([1,-1]); PAULIS=[I,X,Y,Z]; PLABEL='IXYZ'
pauli_mats=[]; pauli_coeff=[]; pauli_weight=[]; pauli_labels=[]
for idx in range(4**q):
    digits=[]; t=idx
    for _ in range(q): digits.append(t%4); t//=4
    P=np.array([[1]],complex)
    for digit in reversed(digits): P=np.kron(P,PAULIS[digit])
    c=np.trace(P@A).real/D
    if abs(c)>1e-11:
        pauli_mats.append(P); pauli_coeff.append(c); pauli_weight.append(sum(x!=0 for x in digits)); pauli_labels.append(''.join(PLABEL[x] for x in reversed(digits)))
pauli_mats=np.asarray(pauli_mats); pauli_coeff=np.asarray(pauli_coeff); pauli_weight=np.asarray(pauli_weight)
print(f"Exact decomposition: {len(pauli_coeff)} nonzero Pauli terms; {sum(pauli_weight>0)} measured terms")

def ry(theta):
    c,s=np.cos(theta/2),np.sin(theta/2); return np.array([[c,-s],[s,c]])

def apply_1q(state, gate, wire):
    shaped=state.reshape([2]*q); shaped=np.moveaxis(shaped, wire, 0).reshape(2,-1)
    shaped=gate@shaped; return np.moveaxis(shaped.reshape([2]+[2]*(q-1)),0,wire).reshape(-1)

def apply_cnot(state, control, target):
    out=state.copy()
    for i in range(D):
        # reshape convention: wire 0 is the most-significant tensor axis
        cb=(i>>(q-1-control))&1; tb=(i>>(q-1-target))&1
        if cb and tb==0:
            j=i^(1<<(q-1-target)); out[i],out[j]=state[j],state[i]
    return out

REPS=4; NPARAM=q*(REPS+1); CNOTS=q*REPS; ONE_Q=NPARAM
def ansatz_state(theta):
    state=np.zeros(D,complex); state[0]=1; k=0
    for layer in range(REPS+1):
        for wire in range(q): state=apply_1q(state,ry(theta[k]),wire); k+=1
        if layer<REPS:
            for wire in range(q): state=apply_cnot(state,wire,(wire+1)%q)
    return state

def exact_energy(theta, operator=A):
    psi=ansatz_state(theta); return float(np.vdot(psi,operator@psi).real)

print(f"Ansatz: {NPARAM} RY parameters, {CNOTS} logical CNOTs, {ONE_Q} one-qubit rotations")


Exact decomposition: 78 nonzero Pauli terms; 77 measured terms
Ansatz: 25 RY parameters, 20 logical CNOTs, 25 one-qubit rotations


In [4]:
@dataclass(frozen=True)
class MeasurementConfig:
    name: str
    shots: int|None
    readout_error: float=0.0
    oneq_depol: float=0.0
    twoq_depol: float=0.0

EXACT=MeasurementConfig('exact',None)
SHOT=MeasurementConfig('shots_4096',4096)
NOISY=MeasurementConfig('shots_4096+noise',4096,readout_error=0.01,oneq_depol=0.0005,twoq_depol=0.008)

def sampled_pauli_energy(theta, cfg, rng, operator_coeff=pauli_coeff, operator_mats=pauli_mats, operator_weight=pauli_weight):
    psi=ansatz_state(theta)
    means=np.einsum('i,kij,j->k',psi.conj(),operator_mats,psi,optimize=True).real.clip(-1,1)
    if cfg.shots is None: return float(operator_coeff@means), 0.0
    # Approximate local noise: readout attenuates measured parity; gate depolarization
    # attenuates nonidentity observables according to a conservative circuit survival factor.
    survival=(1-2*cfg.readout_error)**operator_weight
    survival*=((1-cfg.oneq_depol)**(ONE_Q*operator_weight/q))*((1-cfg.twoq_depol)**(CNOTS*operator_weight/q))
    noisy_means=(means*survival).clip(-1,1)
    measured=np.flatnonzero(operator_weight>0); identity=np.flatnonzero(operator_weight==0)
    alloc=np.zeros(len(means),int)
    # coefficient-weighted allocation, minimum two shots per measured term
    w=np.abs(operator_coeff[measured])*np.sqrt(np.maximum(1-noisy_means[measured]**2,0.02))
    base=2; remaining=max(0,cfg.shots-base*len(measured)); raw=remaining*w/w.sum()
    alloc[measured]=base+np.floor(raw).astype(int)
    leftover=cfg.shots-int(alloc.sum())
    if leftover>0: alloc[measured[np.argsort(raw-np.floor(raw))[-leftover:]]]+=1
    estimates=noisy_means.copy()
    for k in measured:
        plus=rng.binomial(alloc[k],(1+noisy_means[k])/2); estimates[k]=2*plus/alloc[k]-1
    estimates[identity]=means[identity]
    variance=float(np.sum(operator_coeff[measured]**2*(1-noisy_means[measured]**2)/alloc[measured]))
    return float(operator_coeff@estimates), variance

# Sanity check: exact Pauli reconstruction equals the matrix expectation.
tt=np.random.default_rng(1).uniform(-np.pi,np.pi,NPARAM)
assert abs(sampled_pauli_energy(tt,EXACT,np.random.default_rng(2))[0]-exact_energy(tt))<1e-9


## 3. Fair optimizer benchmark

Every optimizer starts from the same point for a given seed and receives at most `MAX_EVALS` calls to the same scalar objective. SPSA uses two calls per update; Adam uses the same simultaneous-perturbation gradient so that a 25-parameter gradient does not secretly cost 50 circuit evaluations. COBYLA and Nelder–Mead use SciPy's function-call cap. Bounds are enforced by wrapping angles to $[-\pi,\pi)$.

The compact default (`3` seeds, `300` evaluations) is designed to run on a laptop. Increase `SEEDS`, `MAX_EVALS`, and `SHOTS` for publication-grade confidence intervals. Hardware cost is reported as total shots and an approximate number of executed CNOTs (`shots × logical CNOTs`); transpilation will change this.


In [5]:
SEEDS=[11,29,47]
MAX_EVALS=300
OPTIMIZERS=['COBYLA','Nelder-Mead','SPSA','Adam-SPSA']
CONFIGS=[EXACT,SHOT,NOISY]

def wrap(x): return (np.asarray(x)+np.pi)%(2*np.pi)-np.pi

def basis_initial_parameters(index, rng, scale=0.08):
    # Physics-informed start: prepare a charge basis state in the final RY layer.
    x=rng.normal(0,scale,NPARAM)
    bits=[(index>>(q-1-wire))&1 for wire in range(q)]
    x[-q:]+=np.pi*np.asarray(bits)
    return wrap(x)

class BudgetObjective:
    def __init__(self,cfg,seed,extra=None):
        self.cfg=cfg; self.rng=np.random.default_rng(seed); self.calls=0; self.history=[]; self.variances=[]; self.extra=extra
    def __call__(self,x):
        if self.calls>=MAX_EVALS: raise StopIteration
        e,var=sampled_pauli_energy(wrap(x),self.cfg,self.rng)
        if self.extra is not None: e2,v2=self.extra(wrap(x),self.rng,self.cfg); e+=e2; var+=v2
        self.calls+=1; self.history.append(e); self.variances.append(var); return e

def run_optimizer(name,obj,x0,seed):
    rng=np.random.default_rng(seed+9000); x=wrap(x0.copy()); best_x=x.copy(); best=np.inf
    if name in ('COBYLA','Nelder-Mead'):
        try:
            method=name; opts={'maxiter':MAX_EVALS,'disp':False}
            if name=='COBYLA': opts.update({'rhobeg':0.35,'tol':1e-3,'catol':0})
            else: opts.update({'maxfev':MAX_EVALS,'xatol':1e-3,'fatol':1e-4,'adaptive':True})
            res=minimize(obj,x,method=method,options=opts); x=wrap(res.x)
        except StopIteration: pass
        if obj.history: best=int(np.argmin(obj.history))
        return x
    a,c,A0,alpha,gamma=(0.12,0.16,20,0.602,0.101)
    m=np.zeros_like(x); v=np.zeros_like(x)
    k=0
    while obj.calls+2<=MAX_EVALS:
        ak=a/(k+1+A0)**alpha; ck=c/(k+1)**gamma; delta=rng.choice([-1.,1.],len(x))
        try: yp=obj(x+ck*delta); ym=obj(x-ck*delta)
        except StopIteration: break
        g=(yp-ym)/(2*ck)*delta
        if name=='Adam-SPSA':
            m=.9*m+.1*g; v=.999*v+.001*g*g
            step=ak*(m/(1-.9**(k+1)))/(np.sqrt(v/(1-.999**(k+1)))+1e-8)
        else: step=ak*g
        x=wrap(x-step); k+=1
    return x

def benchmark_vqe():
    rows=[]; traces={}
    for cfg in CONFIGS:
      for seed in SEEDS:
        # The n=0 charge state has embedded index N; unlike |00000>, it is not padding.
        x0=basis_initial_parameters(N,np.random.default_rng(seed),scale=.10)
        for opt in OPTIMIZERS:
            obj=BudgetObjective(cfg,seed*100+OPTIMIZERS.index(opt))
            t=time.perf_counter(); x=run_optimizer(opt,obj,x0,seed); elapsed=time.perf_counter()-t
            # Fair final score: noiseless energy of returned parameters.
            score=exact_energy(x); psi=ansatz_state(x); fidelity=float(abs(np.vdot(vectors[:,0],psi[:d]))**2)
            rows.append(dict(task='VQE',config=cfg.name,seed=seed,optimizer=opt,calls=obj.calls,total_shots=obj.calls*(cfg.shots or 0),
                             energy=score,error_mhz=1000*abs(score-E0_exact),fidelity=fidelity,
                             mean_estimator_variance=float(np.mean(obj.variances)),seconds=elapsed,logical_cnot_shots=obj.calls*(cfg.shots or 0)*CNOTS))
            traces[(cfg.name,seed,opt)]=obj.history
            print(f"{cfg.name:18s} seed={seed:2d} {opt:11s}: error={1000*abs(score-E0_exact):8.2f} MHz, calls={obj.calls}")
    return rows,traces

vqe_rows,vqe_traces=benchmark_vqe()


exact              seed=11 COBYLA     : error= 2126.24 MHz, calls=300
exact              seed=11 Nelder-Mead: error=17111.25 MHz, calls=300
exact              seed=11 SPSA       : error=27629.17 MHz, calls=300
exact              seed=11 Adam-SPSA  : error= 4525.33 MHz, calls=300
exact              seed=29 COBYLA     : error= 2012.23 MHz, calls=300
exact              seed=29 Nelder-Mead: error=14230.94 MHz, calls=300
exact              seed=29 SPSA       : error=52919.00 MHz, calls=300
exact              seed=29 Adam-SPSA  : error= 4181.87 MHz, calls=300
exact              seed=47 COBYLA     : error= 2006.13 MHz, calls=300
exact              seed=47 Nelder-Mead: error=16215.47 MHz, calls=300
exact              seed=47 SPSA       : error=27538.94 MHz, calls=300
exact              seed=47 Adam-SPSA  : error= 5174.95 MHz, calls=300
shots_4096         seed=11 COBYLA     : error=11904.38 MHz, calls=166
shots_4096         seed=11 Nelder-Mead: error=17881.33 MHz, calls=300
shots_4096         s

In [6]:
# Aggregate without pandas so the notebook has few dependencies.
def aggregate(rows,key='error_mhz'):
    out=[]
    for cfg in [c.name for c in CONFIGS]:
      for opt in OPTIMIZERS:
        vals=np.array([r[key] for r in rows if r['config']==cfg and r['optimizer']==opt])
        if len(vals): out.append(dict(config=cfg,optimizer=opt,mean=float(vals.mean()),std=float(vals.std(ddof=1)) if len(vals)>1 else 0,median=float(np.median(vals))))
    return out

vqe_summary=aggregate(vqe_rows)
fig=go.Figure()
for cfg in [c.name for c in CONFIGS]:
    rr=[r for r in vqe_summary if r['config']==cfg]
    fig.add_bar(name=cfg,x=[r['optimizer'] for r in rr],y=[r['mean'] for r in rr],error_y=dict(type='data',array=[r['std'] for r in rr]))
fig.update_layout(barmode='group',title='VQE ground-energy error (mean ± sample SD over seeds)',xaxis_title='Optimizer',yaxis_title='|E−E₀| [MHz]')
show_figure(fig)

fig=make_subplots(rows=1,cols=3,subplot_titles=[c.name for c in CONFIGS])
for col,cfg in enumerate([c.name for c in CONFIGS],1):
  for opt in OPTIMIZERS:
    curves=[]
    for seed in SEEDS:
      h=np.asarray(vqe_traces[(cfg,seed,opt)]); curves.append(np.minimum.accumulate(h)-E0_exact)
    m=min(map(len,curves)); arr=np.array([x[:m] for x in curves]); med=np.median(arr,axis=0)
    fig.add_trace(go.Scatter(x=np.arange(1,m+1),y=1000*np.abs(med),name=opt,legendgroup=opt,showlegend=(col==1)),row=1,col=col)
fig.update_yaxes(type='log',title_text='median best-so-far error [MHz]',row=1,col=1); fig.update_xaxes(title_text='objective calls')
fig.update_layout(title='VQE convergence under equal objective-call budgets',height=430); show_figure(fig)


## 4. VQD benchmark

VQD minimizes $F_1(\theta)=\langle H\rangle+\beta|\langle\psi_0|\psi(\theta)\rangle|^2$. To isolate excited-state optimizer behavior, this benchmark uses the exact ground eigenvector embedded in five qubits as the deflation reference; on hardware replace it with the best independently prepared VQE ground state. The overlap is sampled as a Bernoulli projector measurement using 20% of the configured shot budget, while 80% is used for the Hamiltonian. This is an optimistic abstraction: a destructive SWAP test or compute–uncompute circuit adds depth and device-specific noise.


In [7]:
ground_ref=np.zeros(D,complex); ground_ref[:d]=vectors[:,0]
BETA=1.5*(E1_exact-E0_exact)
VQD_OPTS=['COBYLA','SPSA','Adam-SPSA']

def make_vqd_extra(ground,beta):
  def extra(theta,rng,cfg):
    psi=ansatz_state(theta); p=float(abs(np.vdot(ground,psi))**2)
    if cfg.shots is None: return beta*p,0.0
    s=max(64,int(.2*cfg.shots)); # overlap share; Hamiltonian sampler still uses configured total, so cost accounting below is conservative
    # Gate noise pushes the overlap toward the maximally mixed value.
    surv=(1-cfg.twoq_depol)**(2*CNOTS)*(1-cfg.oneq_depol)**(2*ONE_Q)
    p_noisy=(surv*p+(1-surv)/D)*(1-2*cfg.readout_error)+cfg.readout_error
    phat=rng.binomial(s,np.clip(p_noisy,0,1))/s
    return beta*phat,beta*beta*p_noisy*(1-p_noisy)/s
  return extra

def benchmark_vqd():
 rows=[]; traces={}
 for cfg in CONFIGS:
  for seed in SEEDS:
   # Charge-odd initialization targets the first excited parity sector, plus a seed perturbation.
   target=np.zeros(D); target[:d]=n*vectors[:,0]; target/=np.linalg.norm(target)
   best_init=None; best_f=-np.inf
   for _ in range(40):
    # The odd first excitation is dominated by charges adjacent to n=0.
    charge_index=N-1 if _%2==0 else N+1
    z=basis_initial_parameters(charge_index,np.random.default_rng(seed+_),scale=.18)
    f=abs(np.vdot(target,ansatz_state(z)))
    if f>best_f: best_f,best_init=f,z
   for opt in VQD_OPTS:
    obj=BudgetObjective(cfg,seed*200+VQD_OPTS.index(opt),extra=make_vqd_extra(ground_ref,BETA))
    t=time.perf_counter(); x=run_optimizer(opt,obj,best_init,seed+17); elapsed=time.perf_counter()-t
    psi=ansatz_state(x); energy=float(np.vdot(psi,A@psi).real); overlap=float(abs(np.vdot(ground_ref,psi))**2)
    f1=float(abs(np.vdot(vectors[:,1],psi[:d]))**2); f01=energy-E0_exact
    shots_per_call=0 if cfg.shots is None else int(1.2*cfg.shots)
    rows.append(dict(task='VQD',config=cfg.name,seed=seed,optimizer=opt,calls=obj.calls,total_shots=obj.calls*shots_per_call,
                     energy=energy,error_mhz=1000*abs(energy-E1_exact),f01_error_mhz=1000*abs(f01-f01_exact),overlap=overlap,
                     fidelity=f1,mean_estimator_variance=float(np.mean(obj.variances)),seconds=elapsed,
                     logical_cnot_shots=obj.calls*shots_per_call*CNOTS))
    traces[(cfg.name,seed,opt)]=obj.history
    print(f"{cfg.name:18s} seed={seed:2d} {opt:11s}: E1 error={1000*abs(energy-E1_exact):8.2f} MHz, F1={f1:.3f}")
 return rows,traces

vqd_rows,vqd_traces=benchmark_vqd()


exact              seed=11 COBYLA     : E1 error=  796.15 MHz, F1=0.822
exact              seed=11 SPSA       : E1 error=20243.89 MHz, F1=0.520
exact              seed=11 Adam-SPSA  : E1 error=  275.93 MHz, F1=0.588
exact              seed=29 COBYLA     : E1 error=  796.15 MHz, F1=0.822
exact              seed=29 SPSA       : E1 error=22680.16 MHz, F1=0.085
exact              seed=29 Adam-SPSA  : E1 error=  247.18 MHz, F1=0.574
exact              seed=47 COBYLA     : E1 error= 1763.72 MHz, F1=0.543
exact              seed=47 SPSA       : E1 error=19279.64 MHz, F1=0.003
exact              seed=47 Adam-SPSA  : E1 error=  938.03 MHz, F1=0.395
shots_4096         seed=11 COBYLA     : E1 error= 8328.10 MHz, F1=0.320
shots_4096         seed=11 SPSA       : E1 error=23075.00 MHz, F1=0.326
shots_4096         seed=11 Adam-SPSA  : E1 error=  665.61 MHz, F1=0.571
shots_4096         seed=29 COBYLA     : E1 error=10473.32 MHz, F1=0.327
shots_4096         seed=29 SPSA       : E1 error=28691.11 MHz, F

In [8]:
vqd_summary=[]
for cfg in [c.name for c in CONFIGS]:
 for opt in VQD_OPTS:
  rr=[r for r in vqd_rows if r['config']==cfg and r['optimizer']==opt]
  vals=np.array([r['error_mhz'] for r in rr]); fids=np.array([r['fidelity'] for r in rr])
  vqd_summary.append(dict(config=cfg,optimizer=opt,error_mean=float(vals.mean()),error_std=float(vals.std(ddof=1)),fidelity_mean=float(fids.mean())))
fig=make_subplots(rows=1,cols=2,subplot_titles=['First-excited energy error','First-excited fidelity'])
for cfg in [c.name for c in CONFIGS]:
 rr=[r for r in vqd_summary if r['config']==cfg]
 fig.add_trace(go.Bar(name=cfg,x=[r['optimizer'] for r in rr],y=[r['error_mean'] for r in rr],error_y=dict(type='data',array=[r['error_std'] for r in rr]),legendgroup=cfg),row=1,col=1)
 fig.add_trace(go.Bar(name=cfg,x=[r['optimizer'] for r in rr],y=[r['fidelity_mean'] for r in rr],legendgroup=cfg,showlegend=False),row=1,col=2)
fig.update_yaxes(title_text='|E−E₁| [MHz]',row=1,col=1); fig.update_yaxes(title_text='mean fidelity',range=[0,1],row=1,col=2)
fig.update_layout(barmode='group',title='VQD accuracy over repeated seeds',height=430); show_figure(fig)


## 5. Ranked results and hardware cost

The next cell prints compact rankings. The final score always uses an exact expectation of the returned parameters, avoiding the misleading practice of selecting the lowest noisy sample. A real experiment should reserve an independent, larger-shot validation allocation for the final few candidates.


In [9]:
def print_ranking(rows,opts,label):
 print('\n'+label)
 print(f"{'configuration':20s} {'optimizer':12s} {'err MHz':>10s} {'SD':>9s} {'fid':>8s} {'calls':>7s} {'shots':>12s}")
 for cfg in [c.name for c in CONFIGS]:
  temp=[]
  for opt in opts:
   rr=[r for r in rows if r['config']==cfg and r['optimizer']==opt]; err=np.array([r['error_mhz'] for r in rr])
   temp.append((err.mean(),cfg,opt,err.std(ddof=1),np.mean([r['fidelity'] for r in rr]),np.mean([r['calls'] for r in rr]),np.mean([r['total_shots'] for r in rr])))
  for mean,cfg,opt,sd,fid,calls,shots in sorted(temp): print(f"{cfg:20s} {opt:12s} {mean:10.2f} {sd:9.2f} {fid:8.3f} {calls:7.0f} {shots:12.0f}")
print_ranking(vqe_rows,OPTIMIZERS,'VQE ranking')
print_ranking(vqd_rows,VQD_OPTS,'VQD ranking')



VQE ranking
configuration        optimizer       err MHz        SD      fid   calls        shots
exact                COBYLA          2048.20     67.66    0.691     300            0
exact                Adam-SPSA       4627.38    504.34    0.662     300            0
exact                Nelder-Mead    15852.55   1474.05    0.305     300            0
exact                SPSA           36029.04  14627.21    0.223     300            0
shots_4096           Adam-SPSA       5169.83    724.42    0.645     300      1228800
shots_4096           COBYLA         10527.65   1896.82    0.489     161       660821
shots_4096           Nelder-Mead    16473.37   1704.52    0.294     300      1228800
shots_4096           SPSA           29143.97   5628.58    0.107     300      1228800
shots_4096+noise     Adam-SPSA       5652.46    742.33    0.628     300      1228800
shots_4096+noise     COBYLA         12572.32   2833.66    0.443     146       599381
shots_4096+noise     Nelder-Mead    16524.21   1703.

## 6. Interpretation, limitations, and quantum-hardware recommendations

### What can be concluded

- Compare optimizers **within each measurement configuration**. The noisy objective is biased by the deliberately simple channel model, so its absolute energy is not a device prediction.
- SPSA and Adam-SPSA spend two circuit-energy evaluations per update independent of parameter count. This is usually preferable to coordinate finite differences on hardware. COBYLA is a useful low-dimensional baseline; Nelder–Mead often consumes its budget building a simplex before improving all 25 parameters.
- Seed-to-seed spread is operationally important. Select an optimizer using median/error bars, not one lucky run.
- VQD is harder than VQE because overlap estimation adds circuits, shots, depth, and a penalty hyperparameter. A penalty around 1.2–2 times a conservative spectral-gap bound is a sensible sweep; too small leaks into the ground state, too large amplifies overlap noise.

### Simulator limitations

1. The single transmon is reduced to one effective charge mode. Coupled modes, flux dependence, leakage outside the cutoff, pulse calibration, drift, crosstalk, and non-Markovian effects are absent.
2. “Gate noise” here attenuates Pauli expectations with a scalar survival model. It is not Aer density-matrix simulation and is not calibrated to a backend. Readout error is symmetric and unmitigated.
3. Pauli terms are sampled independently. Qubit-wise commuting grouping, covariance, basis-change gates, and transpiler routing are omitted. Consequently, shot/circuit counts are conservative in some places and optimistic in others.
4. The VQD overlap sampler is an optimistic Bernoulli abstraction. Actual compute–uncompute or SWAP-test circuits can be much deeper.
5. Exact scoring of returned parameters is available only in simulation. Hardware needs a held-out validation budget.

### Recommended hardware workflow

1. Transpile once for the chosen backend/layout and record physical two-qubit count/depth. Use dynamical decoupling only after backend-specific validation.
2. Start with SPSA (roughly `a=0.05–0.2`, `c=0.05–0.2`, calibration over 10–25 trial pairs) and 2–5 random starts. Recalibrate if the circuit or shot count changes. Adam-SPSA is worth testing but can be sensitive to biased noise.
3. Use 1k–4k total shots per energy early, then 8k–32k for the final 10–20 iterations and held-out validation. Allocate shots by coefficient/empirical variance and group commuting Paulis.
4. Apply measurement-error mitigation first. Add zero-noise extrapolation or probabilistic methods only if their sampling overhead is justified by validation data.
5. For VQD, prepare the ground reference independently, sweep $\beta$, monitor overlap and physical energy separately, and budget the overlap circuits explicitly. Subspace expansion or quantum-number/parity-preserving ansätze may be more robust than deep projector circuits.
6. Report wall time, queue time, circuits, shots, post-mitigation uncertainty, and all failed/restarted runs—not only the best energy.

### Scaling up this notebook

For stronger statistics, set `SEEDS = list(range(10))`, `MAX_EVALS = 500–1500`, and add shot configurations at 1,024, 8,192, and 32,768. On Qiskit hardware, replace `sampled_pauli_energy` with Backend/Runtime Estimator calls while retaining `BudgetObjective`, the seed loops, held-out scoring, and cost accounting.


In [10]:
# Machine-readable result export (optional). In hosted notebooks this writes beside the notebook.
results={'metadata':{'EC_GHz':E_C,'EJ_GHz':E_J,'E0_exact_GHz':E0_exact,'E1_exact_GHz':E1_exact,'f01_exact_GHz':f01_exact,
                     'seeds':SEEDS,'max_evals':MAX_EVALS,'ansatz_parameters':NPARAM,'logical_cnots':CNOTS},
         'vqe':vqe_rows,'vqd':vqd_rows}
with open('transmon_optimizer_results.json','w') as f:
    import json
    json.dump(results,f,indent=2)
print('Wrote transmon_optimizer_results.json')


Wrote transmon_optimizer_results.json
